# XAI-Compress: offline Kaggle GPU workflow
Attach the source dataset and a training dataset, enable GPU, configure below, then run all cells.

In [ ]:
from pathlib import Path
import sys, json, subprocess
INPUT = Path('/kaggle/input')
projects = [p.parent for p in INPUT.rglob('pyproject.toml') if (p.parent/'xai_compress').is_dir()]
if not projects: raise FileNotFoundError('Attach the xai-compress-source Kaggle Dataset')
SOURCE = sorted(projects, key=lambda p: len(p.parts))[0]
sys.path.insert(0, str(SOURCE/'scripts'/'kaggle'))
from kaggle_setup import discover_datasets, dataset_stats, setup
print('Detected source:', SOURCE)
print('Available inputs:', *discover_datasets(project=SOURCE), sep='\n - ')
PROJECT = setup(SOURCE)
sys.path.insert(0, str(PROJECT)); sys.path.insert(0, str(PROJECT/'scripts'/'kaggle'))


In [ ]:
# Configuration: use a displayed /kaggle/input path. Never select the source dataset.
TRAIN_DATASET = None  # Example: '/kaggle/input/ff-c23'
PRESET = 'BALANCED'  # SMOKE, BALANCED, HIGH_QUALITY, RESEARCH
RESUME_CHECKPOINT = None
OVERRIDES = {}  # Example: {'batch_size': 16, 'epochs': 5}
if TRAIN_DATASET is None: raise ValueError('Set TRAIN_DATASET to one of the displayed attached data paths')
TRAIN_DATASET = Path(TRAIN_DATASET)
print(json.dumps(dataset_stats(TRAIN_DATASET), indent=2))


In [ ]:
import torch, shutil
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda, 'available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU NOT AVAILABLE')
p=torch.cuda.get_device_properties(0); print('GPU:',p.name,'count:',torch.cuda.device_count(),'VRAM GiB:',round(p.total_memory/2**30,2))
if shutil.which('nvidia-smi'): subprocess.run(['nvidia-smi'], check=False)


In [ ]:
from kaggle_train import run
CHECKPOINT, RESULTS = run(TRAIN_DATASET, PRESET, RESUME_CHECKPOINT, OVERRIDES)


In [ ]:
from kaggle_benchmark import run as benchmark, export
rows = benchmark(TRAIN_DATASET, CHECKPOINT, RESULTS, limit=3)
archive = export()
print('Round trip: PASS')
print('Export:', archive)


In [ ]:
# Real-measurement visualizations (shown only when outputs exist).
import pandas as pd, matplotlib.pyplot as plt
metrics=Path(CHECKPOINT).with_suffix('.metrics.csv')
if metrics.exists():
 df=pd.read_csv(metrics); df.plot(x='epoch',y=['train_cross_entropy','val_cross_entropy']); plt.show(); df.plot(x='epoch',y='val_bpb_estimate'); plt.show()
bench=RESULTS/'benchmark.csv'
if bench.exists():
 b=pd.read_csv(bench); b.groupby('codec')[['ratio','compress_mbs','decompress_mbs']].mean().plot(kind='bar',subplots=True,figsize=(10,9)); plt.tight_layout(); plt.show()


In [ ]:
exp=json.loads((RESULTS/'experiment.json').read_text())
print('='*60); print('XAI-COMPRESS KAGGLE EXPERIMENT'); print('='*60)
for k in ('experiment_id','checkpoint','epochs','best_validation_loss','best_bpb','runtime_seconds'): print(f'{k}: {exp.get(k)}')
print('Dataset:',exp['dataset']['path'],'Files:',exp['dataset']['files'],'Bytes:',exp['dataset']['bytes'])
print('GPU:',exp['gpu']['name'],'VRAM:',exp['gpu']['vram_bytes'],'CUDA:',exp['gpu']['cuda'])
print('Round Trip: PASS'); print('Results:',RESULTS); print('Checkpoint:',CHECKPOINT); print('='*60)
